# ETL MV Agent-Only 复用既有 run_id 继续实验

这个 notebook 用于复用已经跑完 Feature / Family 的既有 run。它不会重新执行 SQLLoaderAgent、QueryAnalysisRunner、FeatureAgent 或 FamilyAgent，适合从 BatchCluster 开始调试后续流程。

In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

# 如果本地刚修改过 llm_demo 源码，自动 reload 已导入模块。
%load_ext autoreload
%autoreload 2

# 自动定位项目根目录，并加入 Python import 路径。
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / "llm_demo").is_dir() and (path / "tpcds-spark").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT

In [ ]:
from llm_demo.src.agents.batch_cluster_agent import BatchClusterAgent
from llm_demo.src.agents.batch_mv_agent import BatchMVAgent
from llm_demo.src.agents.executor_agent import ExecutorAgent
from llm_demo.src.agents.rewrite_agent import RewriteAgent
from llm_demo.src.agents.self_iteration_agent import SelfIterationAgent
from llm_demo.src.core.artifact_store import ArtifactStore
from llm_demo.src.core.llm_client import LLMClient
from llm_demo.src.core.batch_workflow_runner import BatchWorkflowRunner
from llm_demo.src.core.coverage_summary_builder import CoverageSummaryBuilder


In [ ]:
# 填写你要复用的 run_id。默认使用最近一次全量 Feature/Family run。
run_id = "20260602_112546"

store = ArtifactStore(run_id=run_id, artifact_root=PROJECT_ROOT / "llm_demo" / "artifacts")
llm_client = LLMClient(project_root=PROJECT_ROOT)

sql_manifest_path = store.sql_manifest_path
query_blocks_path = store.query_blocks_path
families_path = store.query_families_path
run_log_path = store.run_log_path

run_id, store.run_dir

In [ ]:
# 检查复用 run 的关键上游 artifact 是否已经存在。
# 如果这里报错，说明该 run 还没有完成对应阶段，不能从 BatchCluster 继续。
required_paths = {
    "sql_manifest": sql_manifest_path,
    "query_blocks": query_blocks_path,
    "query_to_qbs": store.query_to_qbs_path,
    "qb_to_query": store.qb_to_query_path,
    "query_families": families_path,
    "run_log": run_log_path,
}
missing = {name: path for name, path in required_paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(missing)
required_paths

In [ ]:
# 可选：查看上游产物规模，确认不是空文件。
query_blocks = store.read_json(query_blocks_path)["query_blocks"]
families = store.read_json(families_path)["query_families"]
manifest = store.read_json(sql_manifest_path)["queries"]
{
    "query_count": len(manifest),
    "query_block_count": len(query_blocks),
    "family_count": len(families),
}

In [ ]:
# 1. 从已有 QueryBlock / QueryFamily 继续运行 BatchCluster。
#    当前实现会用代码 canonicalize 顶层 batch，避免 LLM 错分 q1/q11/q14a 等 query 后中断。
batches_path = BatchClusterAgent(store, llm_client).run(query_blocks_path, families_path)
batches_path

In [ ]:
# 查看 batch 分布，以及 BatchCluster 是否修正了 LLM 输出。
batches = store.read_json(batches_path)["complexity_batches"]
batch_counts = {batch["batch_type"]: len(batch["query_ids"]) for batch in batches}
last_records = [line for line in run_log_path.read_text(encoding="utf-8").splitlines() if "BatchClusterAgent" in line]
batch_counts, last_records[-1] if last_records else None

In [ ]:
# 2. 定义单 batch dry-run helper：只跑指定 batch 的五步闭环。
#    这比直接 run_all_batches 更适合先验证 batch-1 / batch-2。
#    注意：每个 batch 仍会调用 RewriteAgent 和 BatchMVAgent 的 LLM。

def find_batch(batch_id):
    for batch in store.read_json(batches_path)["complexity_batches"]:
        if batch["batch_id"] == batch_id:
            return batch
    raise ValueError(f"Batch {batch_id} not found")


def run_dry_run_batch(batch_id):
    batch = find_batch(batch_id)
    if not batch.get("query_ids"):
        raise ValueError(f"Batch {batch_id} has no query_ids")

    execution_order_path = store.execution_order_path(batch_id)
    if execution_order_path.exists():
        print(
            f"提示：batch {batch_id} 已存在 execution_order，"
            "ExecutorAgent 会在已有 steps 后追加。若要干净重跑，请换新 run_id 或手动清理该 batch artifacts。"
        )

    rewrite_agent = RewriteAgent(store, llm_client)
    batch_mv_agent = BatchMVAgent(store, llm_client)
    executor_agent = ExecutorAgent(store)
    mv_state_path = store.materialized_mvs_path

    # 1. historical rewrite：只能使用已经物化且可见的历史 MV；batch-1 初始通常为空。
    historical_rewrite_dir = rewrite_agent.run(
        batch_id=batch_id,
        rewrite_stage="historical",
        complexity_batches_path=batches_path,
        sql_manifest_path=sql_manifest_path,
        query_blocks_path=query_blocks_path,
        materialized_mvs_path=mv_state_path,
    )

    # 2. batch-local MV generation：MV Candidate 只能来自当前 batch。
    mv_candidates_path = batch_mv_agent.run(
        batch_id=batch_id,
        complexity_batches_path=batches_path,
        query_blocks_path=query_blocks_path,
        families_path=families_path,
        historical_rewrite_dir=historical_rewrite_dir,
        materialized_mvs_path=mv_state_path,
    )

    # 3. dry-run materialize：当前 demo 不连接 Spark，只把 decision=materialize 且依赖满足的 MV 写入状态。
    mv_state_path = executor_agent.materialize_mvs(
        batch_id=batch_id,
        mv_candidates_path=mv_candidates_path,
        mv_build_sql_path=store.batch_mv_build_sql_path(batch_id),
        materialized_mvs_path=mv_state_path,
    )

    # 4. final rewrite：使用刚更新后的 materialized_mvs.json 重写当前 batch SQL。
    final_rewrite_dir = rewrite_agent.run(
        batch_id=batch_id,
        rewrite_stage="final",
        complexity_batches_path=batches_path,
        sql_manifest_path=sql_manifest_path,
        query_blocks_path=query_blocks_path,
        materialized_mvs_path=mv_state_path,
    )

    # 5. dry-run query order：只输出 SQL 运行顺序，不执行 Spark SQL。
    execution_order_path = executor_agent.run_queries(
        batch_id=batch_id,
        final_rewrite_dir=final_rewrite_dir,
        complexity_batches_path=batches_path,
    )

    return {
        "batch_id": batch_id,
        "query_ids": batch["query_ids"],
        "historical_rewrite_dir": historical_rewrite_dir,
        "mv_candidates_path": mv_candidates_path,
        "materialized_mvs_path": mv_state_path,
        "final_rewrite_dir": final_rewrite_dir,
        "execution_order_path": execution_order_path,
    }


def summarize_batch_outputs(outputs):
    mv_candidates = store.read_json(outputs["mv_candidates_path"])["mv_candidates"]
    execution_order = store.read_json(outputs["execution_order_path"])
    materialized_mvs = store.read_json(outputs["materialized_mvs_path"])["materialized_mvs"]

    final_meta_paths = sorted(outputs["final_rewrite_dir"].glob("*_rewrite_meta.json"))
    rewrite_status_counts = {}
    for meta_path in final_meta_paths:
        status = store.read_json(meta_path).get("status", "unknown")
        rewrite_status_counts[status] = rewrite_status_counts.get(status, 0) + 1

    mv_decision_counts = {}
    for candidate in mv_candidates:
        decision = candidate.get("decision", "unknown")
        mv_decision_counts[decision] = mv_decision_counts.get(decision, 0) + 1

    return {
        "batch_id": outputs["batch_id"],
        "query_count": len(outputs["query_ids"]),
        "query_ids": outputs["query_ids"],
        "mv_candidate_count": len(mv_candidates),
        "mv_decision_counts": mv_decision_counts,
        "materialized_mv_count": len(materialized_mvs),
        "final_rewrite_status_counts": rewrite_status_counts,
        "execution_step_count": len(execution_order.get("steps", [])),
        "execution_order_path": str(outputs["execution_order_path"]),
    }


In [ ]:
# 2.5 可选：清理指定 batch 的 artifacts，复用当前 run_id 干净重跑 batch 阶段。
#    只清理 batch 闭环产生的文件，不清理 SQL manifest、QueryBlock、QueryFamily 和 ComplexityBatch。
#    如果清理上游 batch，建议同时清理已经跑过的下游 batch，例如 batches_to_clean = [1, 2]。

import json
import shutil
from pathlib import Path

# 按需填写；为空时不会清理任何内容。
batches_to_clean = [1, 2]  # 例如：[1] 或 [1, 2]
drop_run_log_entries = True


def clean_batch_artifacts(batch_ids, *, drop_run_log_entries=True):
    batch_ids = sorted({int(batch_id) for batch_id in batch_ids})
    if not batch_ids:
        print("batches_to_clean 为空，未清理任何内容。")
        return {
            "cleaned_batch_ids": [],
            "removed_paths": [],
            "removed_mv_count": 0,
            "removed_run_log_count": 0,
        }

    removed_paths = []

    def remove_path(path):
        path = Path(path)
        if not path.exists():
            return
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        removed_paths.append(str(path))

    for batch_id in batch_ids:
        remove_path(store.batch_mv_candidates_path(batch_id))
        remove_path(store.batch_mv_build_sql_path(batch_id))
        remove_path(store.rewrite_dir(batch_id, "historical"))
        remove_path(store.rewrite_dir(batch_id, "final"))
        remove_path(store.execution_order_path(batch_id))

    # materialized_mvs.json 是跨 batch 可见状态；清理时必须移除来自目标 batch 的 MV。
    removed_mv_ids = set()
    removed_mv_count = 0
    mv_state_path = store.materialized_mvs_path
    if mv_state_path.exists():
        state = store.read_json(mv_state_path)
        kept_mvs = []
        for mv in state.get("materialized_mvs", []):
            source_batch_id = mv.get("source_batch_id")
            available_from_batch = mv.get("available_from_batch")
            if source_batch_id in batch_ids or available_from_batch in batch_ids:
                removed_mv_count += 1
                if mv.get("mv_id"):
                    removed_mv_ids.add(mv["mv_id"])
            else:
                kept_mvs.append(mv)
        if removed_mv_count:
            store.write_json("04_batch_mvs/materialized_mvs.json", {"materialized_mvs": kept_mvs})

        stale_mvs = [
            mv.get("mv_id")
            for mv in kept_mvs
            if set(mv.get("depends_on_mv_ids", [])) & removed_mv_ids
        ]
        if stale_mvs:
            print("提示：仍有未清理 MV 依赖被删除的 MV，请考虑把对应下游 batch 也加入 batches_to_clean：", stale_mvs)

    # coverage / feedback 是派生结果；清理 batch 后需要重新生成。
    remove_path(store.coverage_summary_path)
    remove_path(store.path(f"07_feedback/feedback_rules_{store.run_id}.json"))

    removed_run_log_count = 0
    if drop_run_log_entries and store.run_log_path.exists():
        removed_prefixes = tuple(removed_paths)
        kept_lines = []
        for line in store.run_log_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            record = json.loads(line)
            paths = record.get("input_artifact_paths", []) + record.get("output_artifact_paths", [])
            touches_removed_path = any(
                str(path) == removed_path or str(path).startswith(f"{removed_path}/")
                for path in paths
                for removed_path in removed_prefixes
            )
            if record.get("batch_id") in batch_ids or touches_removed_path:
                removed_run_log_count += 1
            else:
                kept_lines.append(line)
        store.run_log_path.write_text("\n".join(kept_lines) + ("\n" if kept_lines else ""), encoding="utf-8")

    result = {
        "cleaned_batch_ids": batch_ids,
        "removed_paths": removed_paths,
        "removed_mv_count": removed_mv_count,
        "removed_run_log_count": removed_run_log_count,
    }
    print(result)
    return result


clean_batch_artifacts(batches_to_clean, drop_run_log_entries=drop_run_log_entries)


In [ ]:
# 3. 先跑 batch-1 小规模闭环。
#    batch-1 当前只有 2 条 SQL，适合作为后续 rewrite/MV/executor 的 smoke test。
batch_1_outputs = run_dry_run_batch(1)
summarize_batch_outputs(batch_1_outputs)


In [ ]:
# 4. 再跑 batch-2 小规模闭环。
#    batch-2 会读取 batch-1 dry-run materialize 后写入的 materialized_mvs.json。
batch_2_outputs = run_dry_run_batch(2)
summarize_batch_outputs(batch_2_outputs)


In [ ]:
# 5. 可选：生成 coverage summary。
#    跑完 batch-1 / batch-2 后，这里应能看到 mv_candidate / rewrite / execution 的非空统计。
# coverage_path = CoverageSummaryBuilder(store).run()
# store.read_json(coverage_path)


In [ ]:
# 6. 可选：生成 SelfIterationAgent 反馈。
#    建议至少跑完 batch-1 / batch-2 后再执行，这样反馈里会包含 MV / rewrite / executor 证据。
# feedback_path = SelfIterationAgent(store, llm_client).run(store.run_log_path)
# store.read_json(feedback_path)
